In [9]:
!pip install groq -q

In [10]:
import os

import json

import time

import re

import pandas as pd

from groq import Groq

from sklearn.metrics import accuracy_score, f1_score, classification_report

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from rouge_score import rouge_scorer

import nltk

nltk.download('punkt', quiet=True)

True

In [11]:
import os
import glob
from pathlib import Path

# Find test.jsonl inside /kaggle/input/*
test_matches = glob.glob("/kaggle/input/**/test.jsonl", recursive=True)
if not test_matches:
    raise FileNotFoundError("test.jsonl not found under /kaggle/input/**")

TEST_FILE = test_matches[0]
DATA_DIR = str(Path(TEST_FILE).parent)

# Optional sibling files
TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")
MASTER_FILE = os.path.join(DATA_DIR, "master_with_splits.json")

print("DATA_DIR   :", DATA_DIR)
print("TEST_FILE  :", TEST_FILE)
print("TRAIN_FILE :", TRAIN_FILE, "| exists:", os.path.exists(TRAIN_FILE))
print("VAL_FILE   :", VAL_FILE,   "| exists:", os.path.exists(VAL_FILE))
print("MASTER     :", MASTER_FILE, "| exists:", os.path.exists(MASTER_FILE))


DATA_DIR   : /kaggle/input/datasets/anmolchaturvedi12390/dataset
TEST_FILE  : /kaggle/input/datasets/anmolchaturvedi12390/dataset/test.jsonl
TRAIN_FILE : /kaggle/input/datasets/anmolchaturvedi12390/dataset/train.jsonl | exists: True
VAL_FILE   : /kaggle/input/datasets/anmolchaturvedi12390/dataset/val.jsonl | exists: True
MASTER     : /kaggle/input/datasets/anmolchaturvedi12390/dataset/master_with_splits.json | exists: True


In [12]:
# Updated stable Groq model
MODEL_NAME = "llama-3.3-70b-versatile"
FALLBACK_MODEL = "llama-3.1-8b-instant"   # optional fallback in your call_model()

DELAY_SEC = 1.2
MAX_RETRIES = 3
RETRY_BASE_SEC = 1.5

# Kaggle writable output dir
OUTPUT_DIR = "/kaggle/working/baseline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Using model      :", MODEL_NAME)
print("Using TEST_FILE  :", TEST_FILE)
print("Output directory :", OUTPUT_DIR)


Using model      : llama-3.3-70b-versatile
Using TEST_FILE  : /kaggle/input/datasets/anmolchaturvedi12390/dataset/test.jsonl
Output directory : /kaggle/working/baseline_outputs


In [13]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["GROQ_API"] = secrets.get_secret("GROQ_API")
print("GROQ_API loaded:", bool(os.environ.get("GROQ_API")))


GROQ_API loaded: True


In [14]:
# Cell 9: Groq client + sanity checks
from groq import Groq

api_key = os.environ.get("GROQ_API")
if not api_key:
    raise EnvironmentError("GROQ_API not set. Add/load secret first.")

client = Groq(api_key=api_key)

print("Groq client ready.")
print("MODEL_NAME    :", MODEL_NAME)
print("FALLBACK_MODEL:", FALLBACK_MODEL)
print("TEST_FILE     :", TEST_FILE)
print("OUTPUT_DIR    :", OUTPUT_DIR)


Groq client ready.
MODEL_NAME    : llama-3.3-70b-versatile
FALLBACK_MODEL: llama-3.1-8b-instant
TEST_FILE     : /kaggle/input/datasets/anmolchaturvedi12390/dataset/test.jsonl
OUTPUT_DIR    : /kaggle/working/baseline_outputs


In [15]:
# Cell 10: Load test set + derive valid labels from your dataset
def load_test_data(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"Loaded {len(records)} test records from {path}")
    return records

test_records = load_test_data(TEST_FILE)

# Build label sets from master_with_splits if available, else from test set
if os.path.exists(MASTER_FILE):
    with open(MASTER_FILE, "r", encoding="utf-8") as f:
        master_records = json.load(f)
else:
    master_records = test_records

VALID_CATEGORIES = sorted({r.get("category", "").strip() for r in master_records if r.get("category")})
VALID_SEVERITIES = sorted({str(r.get("severity", "")).strip().lower() for r in master_records if r.get("severity")})
VALID_DEPARTMENTS = sorted({
    r.get("routing", {}).get("primary_department", "").strip()
    for r in master_records
    if r.get("routing", {}).get("primary_department")
})

print("VALID_CATEGORIES :", VALID_CATEGORIES)
print("VALID_SEVERITIES :", VALID_SEVERITIES)
print("VALID_DEPARTMENTS:", VALID_DEPARTMENTS)


Loaded 32 test records from /kaggle/input/datasets/anmolchaturvedi12390/dataset/test.jsonl
VALID_CATEGORIES : ['Broken Hospital Bed', 'Crowded Hospital Waiting Room', 'Dirty Hospital Bathroom', 'Empty / Unstaffed Nursing Station', 'Overflowing Hospital Trash (Outside)', 'Rats / Rodent Infestation', 'Torn Hospital Privacy Curtain', 'Unappetizing Hospital Food Tray', 'Unhygienic / Contaminated Hospital Food', 'Water Puddle on Hospital Floor']
VALID_SEVERITIES : ['critical', 'high', 'low', 'medium']
VALID_DEPARTMENTS: ['Administration', 'Dietary', 'Facilities Management', 'Housekeeping', 'Maintenance', 'Nursing', 'Pest Control', 'Waste Management']


In [18]:
# Cell 11: Prompts + API call + parser/extractor
def build_zeroshot_prompt(caption, voice_text):
    return f"""A patient filed a hospital complaint. Here is the information:

Image caption: {caption}
Patient voice complaint: {voice_text}

Extract complaint details and return JSON with:
- category
- severity
- department
- complaint_description
"""

def build_prompted_prompt(caption, voice_text):
    categories_str = "\n".join(f"- {c}" for c in VALID_CATEGORIES)
    departments_str = ", ".join(VALID_DEPARTMENTS)

    return f"""You are a hospital complaint triage system.

Return ONLY a valid JSON object with keys:
category, severity, department, complaint_description

Use only these categories:
{categories_str}

Use only these severity levels:
- low
- medium
- high
- critical

Use only these departments:
{departments_str}

Example 1:
Input:
  Image caption: The hospital bathroom has visible mold and dirty floors.
  Voice complaint: Bathroom is filthy and smells terrible.
Output:
{{"category":"Dirty Hospital Bathroom","severity":"high","department":"Housekeeping","complaint_description":"Patient reported severe unhygienic bathroom conditions including mold, dirt, and foul odor."}}

Example 2:
Input:
  Image caption: Rat seen near hospital food area.
  Voice complaint: I saw a rat near the kitchen corridor.
Output:
{{"category":"Rats / Rodent Infestation","severity":"critical","department":"Pest Control","complaint_description":"Patient observed rodent presence near food area, posing serious hygiene and infection risk."}}

Now process:
Image caption: {caption}
Voice complaint: {voice_text}
"""

def parse_json_output(raw_text):
    if not raw_text:
        return {}, False

    cleaned = re.sub(r"```(?:json)?", "", raw_text, flags=re.IGNORECASE).replace("```", "").strip()

    # outer JSON block
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        chunk = cleaned[start:end+1]
        try:
            return json.loads(chunk), True
        except json.JSONDecodeError:
            pass

    try:
        return json.loads(cleaned), True
    except Exception:
        return {}, False

def extract_fields(parsed):
    pred_cat = str(parsed.get("category", "")).strip()
    pred_sev = str(parsed.get("severity", "")).strip().lower()
    pred_dep = str(parsed.get("department", "")).strip()
    pred_desc = str(parsed.get("complaint_description", "")).strip()
    return pred_cat, pred_sev, pred_dep, pred_desc

def call_model(prompt, record_id, baseline_name):
    models_to_try = [MODEL_NAME]
    if FALLBACK_MODEL and FALLBACK_MODEL != MODEL_NAME:
        models_to_try.append(FALLBACK_MODEL)

    last_err = None

    for model_id in models_to_try:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = client.chat.completions.create(
                    model=model_id,
                    messages=[
                        {"role": "system", "content": "Return valid JSON only. No markdown, no explanation."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0,
                    max_tokens=512,
                )
                if model_id != MODEL_NAME:
                    print(f"  INFO {record_id}: fallback used -> {model_id}")
                return response.choices[0].message.content.strip()

            except Exception as e:
                last_err = e
                msg = str(e).lower()
                retryable = any(k in msg for k in ["429", "rate limit", "timeout", "503", "502", "connection"])
                if retryable and attempt < MAX_RETRIES:
                    time.sleep(RETRY_BASE_SEC * (2 ** (attempt - 1)))
                    continue
                break

    print(f"  ERROR on {record_id} [{baseline_name}]: {last_err}")
    return None


In [19]:
# Cell 12: Baseline runner + metrics + save
def compute_metrics(results, baseline_name):
    gt_cats = [r["gt_category"] for r in results]
    gt_sevs = [r["gt_severity"] for r in results]
    gt_deps = [r["gt_department"] for r in results]

    pr_cats = [r["pred_category"] for r in results]
    pr_sevs = [r["pred_severity"] for r in results]
    pr_deps = [r["pred_department"] for r in results]

    json_valid_count = sum(1 for r in results if r["json_valid"])

    cat_acc = accuracy_score(gt_cats, pr_cats)
    sev_acc = accuracy_score(gt_sevs, pr_sevs)
    dep_acc = accuracy_score(gt_deps, pr_deps)

    cat_f1 = f1_score(gt_cats, pr_cats, average="macro", zero_division=0)
    sev_f1 = f1_score(gt_sevs, pr_sevs, average="macro", zero_division=0)
    dep_f1 = f1_score(gt_deps, pr_deps, average="macro", zero_division=0)

    invalid_cats = sum(1 for x in pr_cats if x not in VALID_CATEGORIES)
    invalid_sevs = sum(1 for x in pr_sevs if x not in VALID_SEVERITIES)
    invalid_deps = sum(1 for x in pr_deps if x not in VALID_DEPARTMENTS)

    smoother = SmoothingFunction().method1
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    bleu_scores, r1, r2, rl = [], [], [], []
    for r in results:
        ref = r["gt_description"].split()
        hyp = r["pred_description"].split()
        if ref and hyp:
            bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smoother))
            rs = scorer.score(r["gt_description"], r["pred_description"])
            r1.append(rs["rouge1"].fmeasure)
            r2.append(rs["rouge2"].fmeasure)
            rl.append(rs["rougeL"].fmeasure)

    def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

    print(f"\nPer-class CATEGORY report [{baseline_name}]")
    print(classification_report(gt_cats, pr_cats, zero_division=0))

    return {
        "baseline": baseline_name,
        "model": MODEL_NAME,
        "total_records": len(results),
        "json_validity_rate": round(json_valid_count / len(results), 4),
        "category_accuracy": round(cat_acc, 4),
        "category_macro_f1": round(cat_f1, 4),
        "severity_accuracy": round(sev_acc, 4),
        "severity_macro_f1": round(sev_f1, 4),
        "department_accuracy": round(dep_acc, 4),
        "department_macro_f1": round(dep_f1, 4),
        "bleu_score": avg(bleu_scores),
        "rouge1": avg(r1),
        "rouge2": avg(r2),
        "rougeL": avg(rl),
        "invalid_category_preds": invalid_cats,
        "invalid_severity_preds": invalid_sevs,
        "invalid_department_preds": invalid_deps,
    }

def run_baseline(test_records, baseline_name, prompt_fn):
    print(f"\n{'='*60}")
    print(f"Running: {baseline_name} ({len(test_records)} records)")
    print(f"{'='*60}")

    results = []
    ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint_{baseline_name.replace(' ', '_')}.csv")

    for i, record in enumerate(test_records, start=1):
        image_id = record.get("image_id", f"row_{i}")
        caption = record.get("input", {}).get("image_caption", record.get("refined_caption", ""))
        voice_text = record.get("voice_text", "")
        gt_cat = record.get("category", "")
        gt_sev = str(record.get("severity", "")).lower()
        gt_dep = record.get("routing", {}).get("primary_department", record.get("department", ""))
        gt_desc = record.get("complaint_description", "")

        print(f"  [{i:02d}/{len(test_records)}] {image_id} ... ", end="", flush=True)

        prompt = prompt_fn(caption, voice_text)
        raw_text = call_model(prompt, image_id, baseline_name)

        parsed, is_valid = parse_json_output(raw_text)
        pr_cat, pr_sev, pr_dep, pr_desc = extract_fields(parsed)

        cat_ok = (pr_cat == gt_cat)
        sev_ok = (pr_sev == gt_sev)
        dep_ok = (pr_dep == gt_dep)

        print(("cat✓" if cat_ok else "cat✗"), "|", ("sev✓" if sev_ok else "sev✗"), "|", ("json✓" if is_valid else "json✗"))

        results.append({
            "baseline": baseline_name,
            "image_id": image_id,
            "gt_category": gt_cat,
            "gt_severity": gt_sev,
            "gt_department": gt_dep,
            "gt_description": gt_desc,
            "pred_category": pr_cat,
            "pred_severity": pr_sev,
            "pred_department": pr_dep,
            "pred_description": pr_desc,
            "json_valid": is_valid,
            "raw_output": raw_text or "",
            "cat_correct": cat_ok,
            "sev_correct": sev_ok,
            "dept_correct": dep_ok,
        })

        # crash-safe checkpoint
        pd.DataFrame(results).to_csv(ckpt_path, index=False)
        time.sleep(DELAY_SEC)

    return results

# Run both baselines
results_zs = run_baseline(test_records, "Baseline-1-ZeroShot", build_zeroshot_prompt)
metrics_zs = compute_metrics(results_zs, "Baseline-1-ZeroShot")

results_pt = run_baseline(test_records, "Baseline-2-Prompted", build_prompted_prompt)
metrics_pt = compute_metrics(results_pt, "Baseline-2-Prompted")

# Save final outputs
all_results = results_zs + results_pt
all_metrics = [metrics_zs, metrics_pt]

results_csv = os.path.join(OUTPUT_DIR, "baseline_results.csv")
metrics_json = os.path.join(OUTPUT_DIR, "baseline_metrics.json")
errors_csv = os.path.join(OUTPUT_DIR, "baseline_errors.csv")

pd.DataFrame(all_results).to_csv(results_csv, index=False)
with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

errors = [r for r in all_results if (not r["cat_correct"]) or (not r["sev_correct"]) or (not r["json_valid"])]
pd.DataFrame(errors).to_csv(errors_csv, index=False)

print("\nSaved:")
print(" -", results_csv)
print(" -", metrics_json)
print(" -", errors_csv)

print("\nSummary:")
print(pd.DataFrame(all_metrics)[[
    "baseline","model","json_validity_rate",
    "category_accuracy","category_macro_f1",
    "severity_accuracy","severity_macro_f1",
    "department_accuracy","department_macro_f1",
    "bleu_score","rouge1","rouge2","rougeL"
]])



Running: Baseline-1-ZeroShot (32 records)
  [01/32] broken_9 ... cat✗ | sev✓ | json✓
  [02/32] broken_1 ... cat✗ | sev✓ | json✓
  [03/32] broken_4 ... cat✗ | sev✓ | json✓
  [04/32] broken_21 ... cat✗ | sev✗ | json✓
  [05/32] crowded_6 ... cat✗ | sev✗ | json✓
  [06/32] crowded_1 ... cat✗ | sev✗ | json✓
  [07/32] crowded_9 ... cat✗ | sev✗ | json✓
  [08/32] crowded_19 ... cat✗ | sev✗ | json✓
  [09/32] dirty_sc_12 ... cat✗ | sev✓ | json✓
  [10/32] dirty_sc_10 ... cat✗ | sev✓ | json✓
  [11/32] dirty_5 ... cat✗ | sev✓ | json✓
  [12/32] nursing_9 ... cat✗ | sev✗ | json✓
  [13/32] nursing_22 ... cat✗ | sev✗ | json✓
  [14/32] nursing_7 ... cat✗ | sev✗ | json✓
  [15/32] trash_11 ... cat✗ | sev✗ | json✓
  [16/32] trash_19 ... cat✗ | sev✗ | json✓
  [17/32] trash_7 ... cat✗ | sev✗ | json✓
  [18/32] rats_sc_3 ... cat✗ | sev✗ | json✓
  [19/32] rats_8 ... cat✗ | sev✗ | json✓
  [20/32] rats_9 ... cat✗ | sev✗ | json✓
  [21/32] rats_sc_5 ... cat✗ | sev✗ | json✓
  [22/32] curtain_15 ... cat✗ | sev✗ | jso